<a href="https://colab.research.google.com/github/lucknera/projekt_interdyscyplinarny/blob/main/oleksii_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor

In [5]:
import pandas as pd


def combine_features():
    mateusz_features = pd.read_csv("features/mateusz-features.csv")
    olek_features = pd.read_csv("features/olek-features.csv")
    combined = pd.merge(olek_features, mateusz_features, on="Area")
    return combined


def read_data(path):
    data = pd.read_csv(path)
    df = pd.merge(data, combine_features(), on="ID")
    df = df.drop(
        columns=["start_date", "end_date", "horizon_lower", "horizon_upper", "Country", "electrical_conductivity"],
        errors="ignore"
    )
    target_cols = [
        "Al",
        "B",
        "Ca",
        "C_organic",
        "C_total",
        "Cu",
        "Fe",
        "Mg",
        "Mn",
        "N",
        "P",
        "K",
        "Na",
        "S",
        "Zn",
    ]
    other_cols = [col for col in df.columns if col not in target_cols]
    new_columns = other_cols + target_cols
    df = df[new_columns]
    return df


In [7]:
df = read_data("features/Train.csv")
df.head()

,ID,Longitude,Latitude,Depth_cm,ph_x,Area,tmin_avg,tmax_avg,prec_avg,B11_x,...,Cu,Fe,Mg,Mn,N,P,K,Na,S,Zn
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,15.240530,25.774622,105.626890,1477.0,...,5.826,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,15.240530,25.774622,105.626890,1513.5,...,4.346,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,21.969696,30.452652,63.580307,2198.0,...,3.657,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,21.969696,30.452652,63.580307,2258.0,...,3.376,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,21.969696,30.452652,63.580307,2258.0,...,3.351,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN


In [8]:
encoder = LabelEncoder()
df['Depth'] = encoder.fit_transform(df['Depth_cm'])
df.drop(columns=['ID', 'Area', 'Depth_cm'], inplace=True)
df.head()

,Longitude,Latitude,ph_x,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,...,Fe,Mg,Mn,N,P,K,Na,S,Zn,Depth
0,37.65189,-3.15440,6.405,15.240530,25.774622,105.626890,1477.0,1153.0,339.5,593.0,...,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN,1
1,37.63612,-3.08585,6.419,15.240530,25.774622,105.626890,1513.5,1084.5,296.0,494.0,...,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN,1
2,39.55580,-2.67218,8.388,21.969696,30.452652,63.580307,2198.0,1346.0,650.0,818.0,...,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN,1
3,39.55477,-2.67196,8.302,21.969696,30.452652,63.580307,2258.0,1621.0,636.0,807.0,...,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN,1
4,39.55477,-2.67196,8.292,21.969696,30.452652,63.580307,2258.0,1621.0,636.0,807.0,...,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN,1


In [9]:
def get_S(row):
    return row['N'] / 12.5

def get_B(row):
    value = row['N'] / 1000
    if row['Ca'] > 1500:
        return value * 0.6
    return value

def get_Zn(row):
    value = row['Mn'] / 10
    if row['ph_x'] > 6.0:
        penalty = (row['ph_x'] - 6.0) * 0.5
        value *= (1 - min(penalty, 0.9))
    return value

def get_Na(row):
    return 0

def get_P(row):
    return 0

In [10]:
df['S'] = df.apply(get_S, axis=1)
df['B'] = df.apply(get_B, axis=1)
df['Zn'] = df.apply(get_Zn, axis=1)
df['Na'] = df.apply(get_Na, axis=1)
df['P'] = df.apply(get_P, axis=1)

In [11]:
df.dropna(axis=0, inplace=True)

In [12]:
target = [
        "Al",
        "B",
        "Ca",
        "Cu",
        "Fe",
        "Mg",
        "Mn",
        "N",
        "P",
        "K",
        "Na",
        "S",
        "Zn",
    ]
x = df.drop(columns=target, errors='ignore')
y_columns = df.columns.difference(x.columns)
y = df[y_columns]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [13]:
x_train.head()

,Longitude,Latitude,ph_x,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,...,Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,C_organic,C_total,Depth
6489,42.73327,9.63641,8.360,11.579545,26.409090,63.800938,2852.5,2296.500000,617.500000,890.333333,...,781.409091,2242.354545,2469.618182,40162.590909,27029.100000,9034.509091,675.527273,24.92,37.41,0
11881,48.70980,-18.27420,4.862,15.196970,25.528410,127.234060,1452.0,588.500000,248.500000,437.500000,...,0.000000,566.727273,593.090909,31748.763636,7640.372727,8651.509091,783.772727,42.07,42.65,0
8746,26.51750,-21.79080,7.003,13.214015,29.979166,32.133892,2864.0,2281.083333,669.259259,842.333333,...,0.000000,473.781818,0.000000,0.000000,11428.600000,44759.450000,0.000000,22.35,26.94,1
8775,26.52350,-21.77120,8.409,13.214015,29.979166,32.133892,3224.0,2577.375000,795.950000,976.333333,...,0.000000,473.781818,0.000000,0.000000,11428.600000,44759.450000,0.000000,14.50,22.31,1
11156,35.28978,-22.85395,6.318,17.833334,30.005682,70.289390,3279.0,2348.250000,559.750000,793.000000,...,623.954545,503.063636,1355.866667,69430.263636,9185.563636,18506.954545,1157.872727,6.25,6.63,0


In [14]:
y_train.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
6489,308.492,0.000822,13740.489,4.572,35.494,530.812,1033.363,127.708,1.37,0,0,0.1096,1.277080
11881,1142.784,0.003250,323.494,2.271,192.366,67.531,96.545,102.805,3.25,0,0,0.2600,10.280500
8746,684.280,0.001158,3938.085,5.797,105.015,1029.506,1604.786,171.618,1.93,0,0,0.1544,8.555157
8775,298.667,0.001020,11769.776,5.077,117.038,896.004,722.193,71.026,1.70,0,0,0.1360,0.710260
11156,510.097,0.000500,638.328,0.566,62.306,47.994,88.265,101.614,0.50,0,0,0.0400,8.545737


In [16]:
models = {}
for col in target:
    model = RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=10)
    model.fit(x_train, y_train[col])
    y_pred = model.predict(x_test)
    rmse = root_mean_squared_error(y_test[col], y_pred)
    models[col] = model
    print(f"{col}: RMSE = {rmse}")

AttributeError: 'RandomForestRegressor' object has no attribute 'to'

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [33]:
class NN_1(nn.Module):
    def __init__(self, input_dim, output_dim=13):
        super(NN_1, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, output_dim)
        )

    def forward(self, x):
        return self.layers(x)


In [31]:
y_train.values.shape

(10732, 13)

In [49]:
model = NN_1(input_dim=x_train.shape[1]).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled = scaler.transform(x_test)

target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train)
y_test_scaled = target_scaler.transform(y_test)

X_train_np = X_train_scaled.astype(np.float32)
X_test_np = X_test_scaled.astype(np.float32)
y_train_np = y_train_scaled.astype(np.float32)
y_test_np = y_test_scaled.astype(np.float32)

X_train_tensor = torch.FloatTensor(X_train_np).to(device)
y_train_tensor = torch.FloatTensor(y_train_np).to(device)
X_val_tensor = torch.FloatTensor(X_test_np).to(device)
y_val_tensor = torch.FloatTensor(y_test_np).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


In [68]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=100):
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        scheduler.step(avg_val_loss)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

    return train_losses, val_losses

train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler)

model.eval()
with torch.no_grad():
    predictions = model(X_val_tensor).cpu().numpy()

Epoch [10/100], Train Loss: 0.1288, Val Loss: 0.1182
Epoch [20/100], Train Loss: 0.1290, Val Loss: 0.1177
Epoch [30/100], Train Loss: 0.1309, Val Loss: 0.1176
Epoch [40/100], Train Loss: 0.1300, Val Loss: 0.1170
Epoch [50/100], Train Loss: 0.1286, Val Loss: 0.1167
Epoch [60/100], Train Loss: 0.1304, Val Loss: 0.1170
Epoch [70/100], Train Loss: 0.1300, Val Loss: 0.1168
Epoch [80/100], Train Loss: 0.1290, Val Loss: 0.1179
Epoch [90/100], Train Loss: 0.1297, Val Loss: 0.1169
Epoch [100/100], Train Loss: 0.1286, Val Loss: 0.1193


In [69]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

y_true = y_test.values.astype(np.float32)
predictions_original = target_scaler.inverse_transform(predictions)

mse = mean_squared_error(y_true, predictions_original)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, predictions_original)
r2 = r2_score(y_true, predictions_original)

print(f"Mean Squared Error: {mse:.4f}")
print(f"Root Mean Squared Error: {rmse:.4f}")
print(f"Mean Absolute Error: {mae:.4f}")
print(f"R² Score: {r2:.4f}")

Mean Squared Error: 42456.2734
Root Mean Squared Error: 206.0492
Mean Absolute Error: 48.5417
R² Score: 0.8809


In [70]:
y_true[:, i]

array([9.454802 , 6.097    , 1.67529  , ..., 9.044306 , 9.1779   ,
       7.7541018], dtype=float32)

In [71]:
for i in range(13):
    mse_i = mean_squared_error(y_true[:, i], predictions_original[:, i])
    rmse_i = np.sqrt(mse_i)
    mae_i = mean_absolute_error(y_true[:, i], predictions_original[:, i])
    r2_i = r2_score(y_true[:, i], predictions_original[:, i])

    print(f"Target {y_test.columns[i]}:")
    print(f"  MSE: {mse_i:.4f}")
    print(f"  RMSE: {rmse_i:.4f}")
    print(f"  MAE: {mae_i:.4f}")
    print(f"  R²: {r2_i:.4f}")
    print()

Target Al:
  MSE: 11596.5596
  RMSE: 107.6873
  MAE: 79.9961
  R²: 0.8924

Target B:
  MSE: 0.0000
  RMSE: 0.0002
  MAE: 0.0001
  R²: 0.9000

Target Ca:
  MSE: 519794.2188
  RMSE: 720.9676
  MAE: 405.1384
  R²: 0.9382

Target Cu:
  MSE: 0.3834
  RMSE: 0.6192
  MAE: 0.4361
  R²: 0.8064

Target Fe:
  MSE: 489.8560
  RMSE: 22.1327
  MAE: 14.7797
  R²: 0.7324

Target K:
  MSE: 6460.8027
  RMSE: 80.3791
  MAE: 41.9454
  R²: 0.8672

Target Mg:
  MSE: 12362.6650
  RMSE: 111.1875
  MAE: 61.4883
  R²: 0.9247

Target Mn:
  MSE: 1219.5249
  RMSE: 34.9217
  MAE: 25.1678
  R²: 0.7501

Target N:
  MSE: 0.0317
  RMSE: 0.1782
  MAE: 0.1161
  R²: 0.9453

Target Na:
  MSE: 0.0000
  RMSE: 0.0000
  MAE: 0.0000
  R²: 1.0000

Target P:
  MSE: 0.0000
  RMSE: 0.0000
  MAE: 0.0000
  R²: 1.0000

Target S:
  MSE: 0.0002
  RMSE: 0.0143
  MAE: 0.0093
  R²: 0.9450

Target Zn:
  MSE: 7.9741
  RMSE: 2.8238
  MAE: 1.9652
  R²: 0.7493



In [66]:
y_test.columns

Index(['Al', 'B', 'Ca', 'Cu', 'Fe', 'K', 'Mg', 'Mn', 'N', 'Na', 'P', 'S',
       'Zn'],
      dtype='object')

In [ ]:
log_transform_cols = ['Ca', 'Mg', 'Cu', 'K', 'Zn']

In [1]:
import torch

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"You are currently using a: {device.upper()}")

You are currently using a: CUDA


In [ ]:
log_transform_models = {}
for col in log_transform_cols:
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    model = TransformedTargetRegressor(regressor=rf, func=np.log1p, inverse_func=np.expm1)
    model.fit(x_train, y_train[col])
    y_pred = model.predict(x_test)
    rmse = root_mean_squared_error(y_test[col], y_pred)
    log_transform_models[col] = model
    print(f"{col}: RMSE = {rmse}")

Ca: RMSE = 646.234551601985
Mg: RMSE = 123.682192371565
Cu: RMSE = 0.6278571865945382
K: RMSE = 83.42459358543753
Zn: RMSE = 2.8049308246466818


Ca Model

In [ ]:
ca = models['Ca']
fi = pd.DataFrame(ca.feature_importances_, index=x_train.columns, columns=['Feature Importance'])
fi.sort_values(by='Feature Importance', ascending=False, inplace=True)
print(f"Feature importance for Ca:")
print(fi.head(10))

Feature importance for Ca:
                               Feature Importance
ph                                       0.703195
C_total                                  0.146023
Longitude                                0.038856
Other fruits, n.e.c.                     0.034474
C_organic                                0.028523
Groundnuts, excluding shelled            0.016229
Latitude                                 0.012123
elevation                                0.005216
tmin_avg                                 0.002569
B8                                       0.001791
